# Accounting Automation Simply Explained
The original rough draft verision can be found on my GitHub here: https://github.com/ScottFrederickSchmidt/AccountingAutomation/blob/main/AccountingAutomation

-------

## Introduction
My simple explained accounting automation program will save a small business <b> over 7% of total accounting costs </b>, according to a senior data science python architect at a fortune 100 company. The project already has the support in some form of three computer science directors at Illinois State University and many senior level programmers at a fortune 100 company. Here is a follow up and finalized verision to clearly explain each individual step on how the accounting automation program works. 

A third of an accountant's day is spent reconciling incorrect, missing or newly posted transactions. This Python pandas dataFrame data science automation program that compares thousands of data transactions to database records which detects unbalanced accounts, new transactions,and human error. This will decrease accounting costs by over 7% yearly according to a Python senior data science architect at a fortune100. This Python accounting automation program will compare thousands of real data transactions and database accounting records, detect new transactions and can increase the efficiency of an accounting team by uo to 30%.

----------
## Competitive Advantage
<i><b>"How will this automation program show a competitive advantage versus SAP ERP?" </b>
    -C.S. director of a major university, PHD in C.S., Former SQL Architect. 

This is the best question for the the project. Mid and small businesses benefit from Python automation scripts at little to no cost. In fact, it has been proven that: <i>"Midsized companies (fewer than 1,000 employees) are more likely to spend around 10 million to 20 million at most, and small companies are not likely to have the need for a fully integrated SAP ERP system unless they have the likelihood of becoming midsized" </i> (Monk, 2009). Yes, 99% of fortune 100 companies use SAP ERP which can handle big data and should be used for any large company. However, Python using pandas can easily handle csv files if less than a million rows with no RAM issues. Therefore, all small businesses should consider using simply programming scripts to help accounting processes. 

In conclusion, yes, if there is a better program with lower costs that program should instead be implemented. But most small companies cannot afford a fancy system that will costs million(s) of dollars.

Sources:
Monk, Ellen F.; Wagner, Brej J. (2009). Concepts in enterprise resource planning (3rd ed.). Boston: Thomson Course Technology. pp. 23–34. ISBN 978-1-4239-0179-2.

---------------------
    
## Save Two Files in CSV format
  <b> Simple! This is the ONLY thing we need do to do.</b>  Python will do the rest! Python can read these excel sheets on the backend. They must be saved in the correct Property/Location spot. To find the location of an Excel file on your computer do the following:
1. Right click on Excel sheet
2. Copy paste the Location
3. End with the excel file title name.
    
In the end, it should look like the below: <br>
    `access=r'C:\Users\scott\OneDrive\Desktop\access.csv'`

In [ ]:
access=r'C:\Users\scott\OneDrive\Desktop\access.csv'
data=r'C:\Users\scott\OneDrive\Desktop\data.csv'

#NOTHING TO BE CHANGED BELOW THIS!
import csv, time
import pandas as pd
from datetime import date

#DISPLAY REPORT ON TODAY'S DATE:
print("Running Budget Analysis report on: ", date.today().strftime("%m-%d-%Y") )
start=time.time()

## View Original Data
Using .head() can show the columns and first five rows of a csv file.

In [ ]:
dataDF=pd.read_csv(data)
dataDF.head()

## View Original Access Data
Using .head() can show the columns and first five rows of a csv file.

In [ ]:
accessDF=pd.csv(access) 
accessDF.head()

## Manipulate Transaction Data by Accounting Subtotals
The pandas library will read the excel csv files by the following code:

In [ ]:
dataDF['Ending Amount'] = dataDF.apply(lambda x: x[' YTD Debits'] - x[' YTD Credits'], axis = 1)
dataDF=dataDF.filter(['GL Account', 'Ending Amount'])
dataList=dataDF.to_dict(orient='list')
dataTotals=dataDF.groupby('GL Account')['Ending Amount'].sum().reset_index()
dataTotals['GL Account'] = dataTotals['GL Account'].str.replace('-', '') #gets rid of hyphens 
dataTotals.head()

## Manipulate Access Data by Accounting Subtotals
This is the shadowbook, accounting  data that are not recorded transactions. Here is a sample of what most Access shadow book data could look like, using of course, fake data which is below:

In [ ]:
accessDF=accessDF.filter(['ACCOUNT ID', 'INVOICE AMOUNT', 'REQ#']) #Only need these 3 columns
accessTotals = accessDF.groupby('ACCOUNT ID')['INVOICE AMOUNT'].sum().reset_index() #Sum totals by account number
accessTotals.head()

## Compare Access and Real Data Subtotals
Do the real transaction accounting object codes and shadowbook accounting subtoals match? Python will soon find out! Note: To all the expert computer scientists out there, I do realize I could of use a join statement. However, in the original draft I did it the probably more complex way using two for loops. A join statement would likely make the code more readable.

In [ ]:
for d in dataTotals['GL Account']: #checks each data account 
    for a in accessTotals['ACCOUNT ID']: # checks each account in Access...
        if d==a: #if Data and Access Account are the same then do the following:
            dRow=dataTotals.loc[dataTotals['GL Account']==d]
            aRow=accessTotals.loc[accessTotals['ACCOUNT ID']==a]
            dSUM=round(float(dRow['Ending Amount'].values), 2) #data Subtotal Number
            aSUM=round(float(aRow['INVOICE AMOUNT'].values),2) #access Subtotal Number
            if  dSUM==aSUM: 
                continue; #nothing else needs to be done.
                #hyphen=a[0:5]+"-"+a[5:7]+"-"+a[7:16]+"-"+a[16:] #adds hyphens to account number, easier to copy/paste
                #print(d, dSUM, "data matches access: ", aSUM, hyphen)
            else:
                hyphens=a[0:5]+"-"+a[5:7]+"-"+a[7:16]+"-"+a[16:] #adds hyphens to account number, easier to copy/paste
                print(a, " Access-----check account----- Data", hyphens)
                #print(dSUM, " data DOES NOT match access ", aSUM, ". Difference is: ", round(float(dSUM-aSUM), 2) 
print("Finished matching data to access accounts in: ", round(time.time()-start, 3), " seconds");
print("----------END SECTION1 ---------- \n")

## Check for Missing D in Req Column
Has something posted? When something has posted as a real data transaction, Access 'Req' column must be updated with a D. Normally, this takes a great deal of time by having to check EVERY individual account. When a new transaction with real money pulls, a D is required in REQ# column in Access.

In [ ]:
accessDF=accessDF.dropna(subset=['REQ#'])
accessDF=accessDF.loc[accessDF['REQ#'].str.contains("D", case=False)] #keeps only access rows with a "D"
#print(accessDF)
accessDsum=accessDF.groupby('ACCOUNT ID')['INVOICE AMOUNT'].sum().reset_index()

#GET EVERY REAL ACCOUNTING NUMBER USED:
dataAccounts=dataDF['GL Account'].unique() #finds only unique accounts
dataAcc=[] #final list

for acc in dataAccounts:
    dataAcc.append(acc.replace('-', '') ) #removes hypens

#CHECK DATA FOR UNBANACED ACCOUNTS:
for account in accessDsum['ACCOUNT ID']: # checks each account in Access...
    aRow=accessDsum.loc[accessDsum['ACCOUNT ID']==account]
    dRow=dataTotals.loc[dataTotals['GL Account']==d]
    accessDtotal=round(float(aRow['INVOICE AMOUNT'].values),2) #access Subtotal Number
    dataTotal=round(float(dRow['Ending Amount'].values), 2) #datat Subtotal Number
    if account in dataAcc: #only need to check for Data Accounts 
        hyphens=account[0:5]+"-"+account[5:7]+"-"+account[7:16]+"-"+account[16:] #see note at end
        if accessDtotal==dataTotal:
            continue; #nothing else is needed to be done.
            #print(account, " is likely correct ", hyphens)
        elif accessDtotal<dataTotal:
            print(account, " Missing Ds in Access in REQ# column: ", hyphens)
        elif accessDtotal>dataTotal:
            print(account, " account likely has too many D in REQ column: ", hyphens)
        else:
            print(account, "Error likely happened on account: ", hyphens) #Note: This should never happen
print("------Completed Section 2--------")
print("Done. Finished comparing Access with data in ", round(time.time()-start, 3), " seconds")

## Special Notes
Line 80 Special Note: The only reason I add back the hyphens is so an accounting assistant can easily copy/paste the account number into a database to retrieve the records. Otherwise, the individual would have to type out the full account number by hand and/or could possibly accidently type the wrong account number. This not only saves time but also makes the work of the accountant more enjoyable. 

## Diclosure
ACCOUNTING AUTOMATION SHOULD NEVER REPLACE AN ACCOUNTANT, CPA OR FINANCIAL ADVISOR: <br>
Python can work at instant speeds to detect new transactions, human error, transpoisitional errors, or missing transactions. It may not detect every error, fraud, or have the mindset of a human. However, Python can help with the following:
1. Compare thousands of transactions between actual processed transactions and accounting books. Are the accounting subtotals balanced?
2. New transactions that have been posted or changed. Has a new transaction been posted? Has an old transaction not been recorded?
3. Finding a transpositional error. Was a transaction recorded as 3832.12 but was actually 3823.12? This is difficult to see with the human eye when there are thousands of transactions.
4. Calculating all possible subtotals. How did our potentially ending balance end at XX.XX?